In [6]:
import sqlite3
import pandas as pd

# Creiamo una connessione a un database in memoria (esiste solo per questa sessione)
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Creiamo il dizionario con dati appositamente "rovinati"
data = {
    'ID_Campione': ['101', '101', '102', '103 ', '104', '105', '106'],
    'Zona': ['North', 'north', 'South_Zone', 'north', 'West', 'East', 'North'],
    'Quota_m': ['1250m', '1250m', '1100ft', '950', '880m', 'unknown', '1050m'],
    'Data_Rilievo': ['2023.05.15', '2023.05.15', '15/05/2023', '2023-06-01', '02/06/2023', '2023.06.05', '2023.06.10'],
    'Note': ['Tracce di Oro (Au)', 'Tracce di Oro (Au)', 'Presenza Argento(Ag)', 'Nessuna nota', 'Rilevato Rame (Cu)', 'Vena (Quartz)', 'Errore coordinate']
}

# Carichiamo i dati in una tabella SQL chiamata 'Rilievi_Grezzi'
df_sporco = pd.DataFrame(data)
df_sporco.to_sql('Rilievi_Grezzi', conn, index=False, if_exists='replace')

print("Database creato con successo! La tabella 'Rilievi_Grezzi' è pronta.")
print(df_sporco.head(7))

Database creato con successo! La tabella 'Rilievi_Grezzi' è pronta.
  ID_Campione        Zona  Quota_m Data_Rilievo                  Note
0         101       North    1250m   2023.05.15    Tracce di Oro (Au)
1         101       north    1250m   2023.05.15    Tracce di Oro (Au)
2         102  South_Zone   1100ft   15/05/2023  Presenza Argento(Ag)
3        103        north      950   2023-06-01          Nessuna nota
4         104        West     880m   02/06/2023    Rilevato Rame (Cu)
5         105        East  unknown   2023.06.05         Vena (Quartz)
6         106       North    1050m   2023.06.10     Errore coordinate


# Esercizio A: Normalizzazione Nomi
La colonna `Zona` ha "North", "north" e "South_Zone".
* **Obiettivo:** Crea una query che restituisca una colonna chiamata `Zona_Pulita` dove tutto è in maiuscolo e "SOUTH_ZONE" diventa semplicemente "SOUTH".

# Esercizio B: Conversione Metri/Piedi
La colonna `Quota_m` ha valori come "1250m", "1100ft" e persino un "unknown".
* **Obiettivo:** Converti tutto in numeri puri (FLOAT). Se trovi "ft", moltiplica per 0.3048. Se trovi "unknown", trasformalo in `NULL`.

# Esercizio C: Chirurgia delle Date
La colonna `Data_Rilievo` mischia il formato `YYYY.MM.DD` con il formato europeo `DD/MM/YYYY`.
* **Obiettivo:** Usa `CASE` e `SUBSTR` per trasformare tutto nello standard `YYYY-MM-DD`.

# Esercizio D: Eliminazione Duplicati
L'ID `101` compare due volte con gli stessi identici dati.
* **Obiettivo:** Scrivi una query che usi `DISTINCT` o `GROUP BY` per mostrare la tabella senza quel duplicato fastidioso.

---

# Come testare le tue soluzioni in Jupyter
Per ogni esercizio, usa questo formato nella cella:

```python
q = """
-- Scrivi qui la tua query SQL
SELECT ...
FROM Rilievi_Grezzi
"""
pd.read_sql_query(q, conn)
```



In [7]:
q = """
SELECT 
    -- Pulizia ID_Campione ('103 ' -> '103')
    TRIM(ID_Campione) AS id, 

    -- Pulizia Zona (-> UPPERCASE / South_Zone --> SOUTH)
    CASE
        WHEN Zona = 'South_Zone' THEN 'SOUTH'
        ELSE UPPER(Zona) 
    END AS zona,

    -- Pulizia Quota_m (ft -> m / STRING --> REAL / unkwnown -> NUL)
    CASE
    --* Gestione piedi (conversione in metri: 1 ft = 0.3048 m)
        WHEN Quota_m LIKE '%ft' THEN 
        CAST(REPLACE(Quota_m, 'ft', '') AS FLOAT) * 0.3048 
   
    --* Gestione metri (es. '1250m')
        WHEN Quota_m LIKE '%m' THEN 
        CAST(REPLACE(Quota_m, 'm', '') AS FLOAT)
    
    --* Gestione valori numerici senza unità (es. '950')
        WHEN Quota_m GLOB '[0-9]*' AND Quota_m NOT LIKE '%[a-z]%' THEN 
        CAST(Quota_m AS FLOAT)
    
    --* Gestione 'unknown' esplicito
        WHEN Quota_m = 'unknown' THEN NULL
    
    --* Qualsiasi altro caso non gestito
    ELSE NULL
    END AS quota_metri,

    -- Pulizia Data_rilievo (-> formato unico YYYY-MM-DD)
    CASE
        --* Formato YYYY.MM.DD (2023.01.11)
        WHEN Data_Rilievo LIKE '%.%' THEN 
        REPLACE(Data_Rilievo, '.', '-')

        --* Formato DD/MM/YYYY (01/11/2023)
        WHEN Data_Rilievo LIKE '%/%' AND LENGTH(Data_Rilievo) = 10 THEN 
            SUBSTR(Data_Rilievo, 7, 4) || '-' ||
            SUBSTR(Data_Rilievo, 4, 2) || '-' ||
            SUBSTR(Data_Rilievo, 1, 2)
            
        --* Qualsiasi altro caso non gestito
        ELSE Data_rilievo
    END AS data_rilievo,



    
    Note
FROM Rilievi_Grezzi
"""
# Eseguiamo la query e leggiamo i risultati in un nuovo DataFrame
df_pulito = pd.read_sql_query(q, conn)

print("\nRisultati della query di pulizia:")
print(df_pulito)

# Chiudiamo la connessione
conn.close()


Risultati della query di pulizia:
    id   zona  quota_metri data_rilievo                  Note
0  101  NORTH      1250.00   2023-05-15    Tracce di Oro (Au)
1  101  NORTH      1250.00   2023-05-15    Tracce di Oro (Au)
2  102  SOUTH       335.28   2023-05-15  Presenza Argento(Ag)
3  103  NORTH       950.00   2023-06-01          Nessuna nota
4  104   WEST       880.00   2023-06-02    Rilevato Rame (Cu)
5  105   EAST          NaN   2023-06-05         Vena (Quartz)
6  106  NORTH      1050.00   2023-06-10     Errore coordinate
